# BG-forecasting — analysis stages on Colab

Runs every analysis stage over runs that have **already been trained** by
`run_on_colab.ipynb`. This notebook never trains. It is the Colab driver for
`RUN/run_experiments_analysis.sh`, plus the two things that script assumes and
Colab does not have: the runs restored from Drive, and a run list pointing at
them.

```
trained cells (Drive)  ->  merge seeds  ->  run list  ->  preflight  ->  stages  ->  Drive
```

---

### What this needs before it will work

**All three seeds of a cell must be finished.** The stages read cross-seed
aggregates, and the paired analyses (transfer, and the shift stage's
transfer-benefit comparison) refuse a cell whose regular and transfer seeds
differ. Cells that are short are reported by the preflight and skipped.

**The raw OhioT1DM data is required**, not just the trained runs: the shift,
persistence and rapid-change stages reconstruct prediction context from the CGM
series. Section 3 stages it exactly as the training notebook does.

**A GPU is not needed.** Every stage is numpy/scipy/matplotlib on the CPU, so
use a CPU runtime and keep your compute units for training.

## 1 · Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- edit these to match your Drive ---------------------------------------
DRIVE_DATA    = "/content/drive/MyDrive/ohiot1dm"      # must contain 2018/ and 2020/
DRIVE_RESULTS = "/content/drive/MyDrive/bg-results"    # where the training notebook mirrored
REPO_DIR      = "/content/BG-forecasting"

SOURCE     = "git"
REPO_URL   = "https://github.com/beatriz-fulgencio/BG-forecasting.git"
BRANCH     = "bench2"
DRIVE_REPO = "/content/drive/MyDrive/BG-forecasting"

SEEDS    = [41, 42, 43]
MODELS   = ["gru", "lstm", "rnn"]
HORIZONS = [15, 30, 45, 60]

# Resample counts. The defaults are the paper's (20000 bootstrap, 10000
# permutations). QUICK cuts them to 2000/1000 for a fast rehearsal -- fine for
# checking that every stage runs, not for numbers you publish.
QUICK = False
# ---------------------------------------------------------------------------

import os, pathlib
for d in (DRIVE_RESULTS, f"{DRIVE_RESULTS}/analysis"):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("Drive ready.")

## 2 · Get the code

In [ ]:
import shutil, subprocess, sys, pathlib

if SOURCE == "git":
    if not pathlib.Path(REPO_DIR).is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("WARNING: git pull --ff-only failed; this checkout may be stale.\n"
                  + (pull.stderr or pull.stdout).strip() + "\n")
elif SOURCE == "drive":
    if not pathlib.Path(REPO_DIR).is_dir():
        shutil.copytree(DRIVE_REPO, REPO_DIR)
else:
    raise ValueError("SOURCE must be 'git' or 'drive'")

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

required = ["RUN/run_experiments_analysis.sh", "RUN/run_dataset_analysis.sh",
            "RUN/_common.sh", "merge_seed_runs.py"]
missing = [f for f in required if not pathlib.Path(f).is_file()]
if missing:
    raise SystemExit("Missing from this checkout:\n  " + "\n  ".join(missing))

head = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %cd %s", "--date=short"],
                      capture_output=True, text=True).stdout.strip()
print(f"Repo ready at {REPO_DIR}" + (f"\n  commit: {head}" if head else ""))

import importlib
for mod in ("benchmark.analysis", "scipy", "sklearn", "matplotlib"):
    importlib.import_module(mod)
print("  analysis dependencies import cleanly.")

## 3 · Stage the OhioT1DM data

The shift, persistence and rapid-change stages read the CGM series directly, so
the raw files must be present even though nothing is trained here. Copied file
by file, so an interrupted copy is repaired rather than skipped.

In [ ]:
import shutil, pathlib

COHORT = {"2018": [559, 563, 570, 575, 588, 591],
          "2020": [540, 544, 552, 567, 584, 596]}

dst = pathlib.Path(REPO_DIR) / "data" / "raw" / "ohiot1dm"
src = pathlib.Path(DRIVE_DATA)
if not src.is_dir():
    raise SystemExit(f"{DRIVE_DATA} not found.")

copied = 0
for release, patients in COHORT.items():
    for mode, suffix in (("train", "training"), ("test", "testing")):
        (dst / release / mode).mkdir(parents=True, exist_ok=True)
        for pid in patients:
            name = f"{pid}-ws-{suffix}.xml"
            target, source = dst / release / mode / name, src / release / mode / name
            if target.is_file() or not source.is_file():
                continue
            shutil.copy2(source, target)
            copied += 1

missing = [f"{r}/{m}/{p}-ws-{s}.xml"
           for r, pats in COHORT.items()
           for m, s in (("train", "training"), ("test", "testing"))
           for p in pats if not (dst / r / m / f"{p}-ws-{s}.xml").is_file()]
print(f"{copied} file(s) copied; {len(sorted(dst.glob('*/*/*.xml')))}/24 staged.")
if missing:
    raise SystemExit("Missing OhioT1DM file(s):\n  " + "\n  ".join(missing))

## 4 · Restore the trained runs from Drive

Pulls back both trees the training notebook writes: the single-seed parents
(`experiments/`) and, if it exists already, the merged three-seed tree
(`experiments_merged/`).

In [ ]:
import pathlib, shutil

EXP_DIR    = pathlib.Path(REPO_DIR) / "results" / "experiments"
MERGED_DIR = pathlib.Path(REPO_DIR) / "results" / "experiments_merged"
EXP_DIR.mkdir(parents=True, exist_ok=True)

restored = 0
for d in sorted(pathlib.Path(f"{DRIVE_RESULTS}/experiments").glob("experiment_*")):
    target = EXP_DIR / d.name
    if not target.exists():
        shutil.copytree(d, target)
        restored += 1

drive_merged = pathlib.Path(f"{DRIVE_RESULTS}/experiments_merged")
if drive_merged.is_dir() and not MERGED_DIR.is_dir():
    shutil.copytree(drive_merged, MERGED_DIR)
    print(f"Merged tree restored from {drive_merged}")

print(f"Restored {restored} experiment dir(s); {len(list(EXP_DIR.glob('experiment_*')))} present.")

# Which cells have all three seeds? The stages need complete cells.
import yaml
done = set()
for resolved in EXP_DIR.glob("*/resolved_config.yaml"):
    if (resolved.parent / "aggregate_metrics.json").is_file():
        try:
            done.add(yaml.safe_load(resolved.read_text())["experiment"]["name"])
        except Exception:
            pass

complete, short = [], []
for model in MODELS:
    for horizon in HORIZONS:
        cell = f"full_{model}_{horizon}min"
        have = [s for s in SEEDS if f"{cell}_seed{s}" in done]
        (complete if len(have) == len(SEEDS) else short).append((cell, have))
print(f"\n{len(complete)}/{len(complete) + len(short)} cells have all {len(SEEDS)} seeds.")
for cell, have in short:
    print(f"  incomplete: {cell}  seeds {have or 'none'}")

## 5 · Merge the seeds

One parent per cell holding all three seeds, with cross-seed means and Student-t
intervals, rebuilt by the benchmark's own aggregation code. Cells that are not
complete are reported and skipped. Safe to re-run: `--force` replaces what is
already there.

In [ ]:
import subprocess, sys, shutil, pathlib

args = [sys.executable, "merge_seed_runs.py",
        "--seeds", *map(str, SEEDS),
        "--models", *MODELS,
        "--horizons", *map(str, HORIZONS),
        "--copy"]

print(subprocess.run(args + ["--dry-run"], cwd=REPO_DIR, capture_output=True, text=True).stdout)
result = subprocess.run(args + ["--force"], cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

merged_cells = sorted(p.name for p in MERGED_DIR.glob("full_*")) if MERGED_DIR.is_dir() else []
print(f"{len(merged_cells)} merged cell(s): {', '.join(merged_cells) or 'none'}")
if not merged_cells:
    raise SystemExit("Nothing merged -- no cell has all three seeds yet.")

## 6 · Write the run list

`RUN/run_experiments_analysis.sh` iterates over `results/full_run_logs/parents.txt`,
which `RUN/run_training.sh` normally writes as it trains. The seed-major path
never produces it, so it is written here, pointing at the merged cells rather
than the single-seed parents — one line per cell, `<model> <horizon> <dir>`.

In [ ]:
import pathlib

parents_path = pathlib.Path(REPO_DIR) / "results" / "full_run_logs" / "parents.txt"
parents_path.parent.mkdir(parents=True, exist_ok=True)

lines = []
for model in MODELS:
    for horizon in HORIZONS:
        cell_dir = MERGED_DIR / f"full_{model}_{horizon}min"
        if cell_dir.is_dir():
            lines.append(f"{model} {horizon} {cell_dir}")

parents_path.write_text("\n".join(lines) + "\n")
print(f"{len(lines)} cell(s) recorded in {parents_path}\n")
print(parents_path.read_text())

## 7 · Preflight

Everything the expensive stages can fail on is knowable in seconds: missing
predictions, missing raw data, a stale run list, a broken regular/transfer seed
pairing. Read the warnings — a cell flagged `unpaired` is silently skipped by
the paired analyses.

In [ ]:
import subprocess, os, sys

env = dict(os.environ, SEEDS=" ".join(map(str, SEEDS)),
           MODELS=" ".join(MODELS), HORIZONS=" ".join(map(str, HORIZONS)))

proc = subprocess.run(["bash", "RUN/run_experiments_analysis.sh", "--check"],
                      cwd=REPO_DIR, env=env, capture_output=True, text=True)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr)
print(f"preflight exit status: {proc.returncode}")

## 8 · The dataset signal table

The shift stage needs the per-patient signal features (train/test Wasserstein-1
shift, sample entropy, autocorrelation). It is a separate driver and must run
first.

In [ ]:
import subprocess, os

proc = subprocess.run(["bash", "RUN/run_dataset_analysis.sh", "signal"],
                      cwd=REPO_DIR, env=env, capture_output=True, text=True)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    print(proc.stderr[-4000:])
    raise SystemExit("signal stage failed; the shift analysis cannot run without it")

## 9 · Run the stages

One stage at a time, mirroring to Drive after each, so a disconnect costs you at
most the stage in flight. Timings below are for 12 cells × 12 patients × 3 seeds
at the default resample counts; `QUICK = True` cuts the resamples to 2000/1000.

| stage | cost |
| --- | --- |
| zone-d, error-range | ~1 min each, all cells |
| rapid-change, figures | ~1–2 min per cell |
| glycemic | ~4 min |
| tsne | ~5 min, one cell |
| persistence | ~4 min, reads every prediction CSV |
| shift | ~5 min per model |
| transfer, stability, horizons, compare, summary | seconds |

In [ ]:
import subprocess, shutil, pathlib, time, os

STAGES = ["zone-d", "error-range", "rapid-change", "figures", "glycemic", "tsne",
          "persistence", "shift", "transfer", "stability", "horizons", "compare", "summary"]

ANALYSIS_DIR = pathlib.Path(REPO_DIR) / "results" / "analysis"

def mirror_analysis():
    dst = pathlib.Path(f"{DRIVE_RESULTS}/analysis")
    if ANALYSIS_DIR.is_dir():
        shutil.copytree(ANALYSIS_DIR, dst, dirs_exist_ok=True)

failed = []
for i, stage in enumerate(STAGES, 1):
    cmd = ["bash", "RUN/run_experiments_analysis.sh", stage] + (["--quick"] if QUICK else [])
    print(f"\n{'='*62}\n[{i}/{len(STAGES)}] {stage}\n{'='*62}", flush=True)
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=REPO_DIR, env=env, capture_output=True, text=True)
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print(proc.stderr[-3000:])
        failed.append(stage)
    mirror_analysis()
    print(f"  {'OK ' if proc.returncode == 0 else 'FAIL'} {stage} in {(time.time()-t0)/60:.1f} min", flush=True)

print("\nFailed stages:", failed if failed else "none")

## 10 · What landed

In [ ]:
import pathlib

print(f"{'stage directory':<34}{'files':>8}")
print("-" * 42)
for d in sorted(ANALYSIS_DIR.iterdir()) if ANALYSIS_DIR.is_dir() else []:
    if d.is_dir():
        print(f"{d.name:<34}{len(list(d.rglob('*'))):>8}")
    else:
        print(f"{d.name:<34}{'(file)':>8}")

print(f"\nMirrored to {DRIVE_RESULTS}/analysis")